# Commissiestructuur & Invloed

**Thema:** Commissies, zetelverdelingen en voortouwcommissies.

**Kernvragen:** Welke vaste en tijdelijke commissies zijn het meest actief (krijgen de meeste zaken/activiteiten toebedeeld) en hoe is het verloop in vaste zetels en vacatures?

Dit notebook verkent o.a. de silver-tabellen `commissie`, `commissie_zetel`, `commissie_zetel_vast_persoon`, `commissie_zetel_vast_vacature` en `activiteit` (voortouwcommissie).

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Onderwerp 1

In dit onderzoek staat de organisatie en invloed van parlementaire commissies centraal. Commissies bereiden veel parlementair werk voor en kunnen daarbij verschillende rollen hebben. De ene commissie is groot en heeft veel leden, terwijl een andere commissie kleiner is of tijdelijk wordt ingesteld. Ook kunnen commissies sterk verschillen in het aantal activiteiten en zaken waarvoor zij het voortouw hebben.
De deelvragen hieronder brengen deze verschillen systematisch in beeld. Daarbij wordt niet alleen gekeken naar aantallen, maar ook naar verloop, samenstelling en de mate waarin activiteiten daadwerkelijk worden uitgevoerd.

In [ ]:
tabellen = query_df("""
  SELECT table_name
  FROM information_schema.tables
  WHERE table_schema = 'silver'
  ORDER BY table_name
""")

aantallen = pd.DataFrame(
    [(t, query_df(f"SELECT COUNT(*) AS n FROM silver.{t}")["n"][0]) for t in tabellen["table_name"]],
    columns=["tabel", "aantal_rijen"],
)
aantallen

## 1. Welke soorten commissies zijn er, en hoeveel zijn nog actief?
We onderzoeken welke soorten commissies er zijn en hoeveel daarvan nog actief zijn. Door de commissies op basis van soort te groeperen, krijgen we inzicht in de structuur van het commissiestelsel. We tellen per soort hoeveel commissies er zijn en gebruiken datum_inactief om te bepalen of een commissie nog actief is. De resultaten laten zien dat sommige commissies geen ingevulde waarde bij soort hebben, waardoor deze categorie niet direct aan een specifiek type kan worden gekoppeld.

In [ ]:
soorten = query_df("""
  SELECT
    soort,
    COUNT(*) AS aantal_commissies,
    COUNT(*) FILTER (WHERE datum_inactief IS NULL) AS nog_actief
  FROM silver.commissie
  GROUP BY soort
  ORDER BY aantal_commissies DESC
""")

soorten.plot.bar(x="soort", y=["aantal_commissies", "nog_actief"], figsize=(8, 4),
                 title="Aantal commissies per soort")
plt.show()
soorten

## 2. Hoe groot zijn de commissies (aantal zetels)?
We onderzoeken welke commissies de meeste zetels hebben. Hiervoor tellen we per commissie het aantal zetels uit commissie_zetel en vergelijken we de commissies met elkaar. Zo wordt duidelijk welke commissies relatief groot zijn en welke kleiner.

In [ ]:
zetels = query_df("""
  SELECT
    c.naam_nl AS naam,
    c.soort,
    COUNT(z.id) AS aantal_zetels
  FROM silver.commissie c
  JOIN silver.commissie_zetel z ON z.commissie_id = c.id
  GROUP BY c.naam_nl, c.soort
  ORDER BY aantal_zetels DESC, c.naam_nl
  LIMIT 20
""")

zetels.sort_values("aantal_zetels").plot.barh(x="naam", y="aantal_zetels", figsize=(10, 7),
                                              title="Top 20 commissies naar aantal zetels", legend=False)
plt.show()

In [ ]:
COMMISSIE_TYPE = """
  CASE
    WHEN c.naam_nl ILIKE '%enquête%' OR c.naam_nl ILIKE '%tijdelijke%' THEN 'tijdelijk'
    WHEN c.naam_nl ILIKE 'vaste commissie%' OR c.naam_nl ILIKE 'algemene commissie%'
         OR c.naam_nl ILIKE 'commissie voor%' THEN 'vast'
    WHEN c.naam_nl ILIKE 'delegatie%' OR c.naam_nl ILIKE '%groep%' THEN 'interparlementair'
    ELSE 'overig'
  END
"""

typen = query_df(f"""
  SELECT
    {COMMISSIE_TYPE} AS commissietype,
    COUNT(*) AS aantal_commissies,
    COUNT(*) FILTER (WHERE c.datum_inactief IS NULL) AS nog_actief
  FROM silver.commissie c
  GROUP BY ALL
  ORDER BY aantal_commissies DESC
""")

typen.plot.bar(x="commissietype", y=["aantal_commissies", "nog_actief"], figsize=(8, 4),
               title="Aantal commissies per afgeleid type")
plt.show()
typen

## 3. Vacatures: waar zijn ze en hoe lang blijven ze open?
We onderzoeken waar de meeste vacatures voorkomen, hoe lang vacatures gemiddeld openstaan en hoe het aantal nieuwe vacatures zich per jaar ontwikkelt. Hiervoor gebruiken we de begin- en einddatum van vacatures. Als tot_en_met leeg is, rekenen we de vacature tot vandaag; hierdoor kunnen oude openstaande vacatures het gemiddelde beïnvloeden.

In [ ]:
vacatures = query_df("""
  SELECT
    c.naam_nl AS naam,
    COUNT(*) AS aantal_vacatures,
    COUNT(*) FILTER (WHERE v.tot_en_met IS NULL) AS nu_open,
    ROUND(AVG(date_diff('day', CAST(v.van AS DATE),
                        COALESCE(CAST(v.tot_en_met AS DATE), current_date))), 0) AS gem_dagen_open
  FROM silver.commissie_zetel_vast_vacature v
  JOIN silver.commissie_zetel z ON z.id = v.commissie_zetel_id
  JOIN silver.commissie c ON c.id = z.commissie_id
  GROUP BY c.naam_nl
  ORDER BY aantal_vacatures DESC, c.naam_nl
  LIMIT 20
""")

vacatures.sort_values("aantal_vacatures").plot.barh(x="naam", y="aantal_vacatures", figsize=(10, 7),
                                                    title="Top 20 commissies met de meeste vacatures", legend=False)
plt.show()
vacatures

In [ ]:
vacatures_per_jaar = query_df("""
  SELECT
    year(CAST(van AS DATE)) AS jaar,
    COUNT(*) AS nieuwe_vacatures
  FROM silver.commissie_zetel_vast_vacature
  WHERE van IS NOT NULL
  GROUP BY jaar
  ORDER BY jaar
""")

vacatures_per_jaar.plot(x="jaar", y="nieuwe_vacatures", marker="o", figsize=(10, 4),
                        title="Nieuwe vacatures per jaar", legend=False)
plt.show()

## 4. Verloop in vaste zetels
We onderzoeken hoeveel leden jaarlijks bij commissies komen en vertrekken en hoe lang leden gemiddeld in een commissie zitten. Hiervoor gebruiken we van als startdatum en tot_en_met als einddatum van een zitting. Door de zittingsduur per commissie te vergelijken, zien we waar relatief veel wisselingen plaatsvinden.

In [ ]:
verloop = query_df("""
  WITH start AS (
    SELECT year(CAST(van AS DATE)) AS jaar, COUNT(*) AS n
    FROM silver.commissie_zetel_vast_persoon WHERE van IS NOT NULL GROUP BY 1
  ),
  einde AS (
    SELECT year(CAST(tot_en_met AS DATE)) AS jaar, COUNT(*) AS n
    FROM silver.commissie_zetel_vast_persoon WHERE tot_en_met IS NOT NULL GROUP BY 1
  )
  SELECT
    COALESCE(start.jaar, einde.jaar) AS jaar,
    COALESCE(start.n, 0) AS nieuwe_leden,
    COALESCE(einde.n, 0) AS vertrokken_leden
  FROM start FULL OUTER JOIN einde ON start.jaar = einde.jaar
  ORDER BY jaar
""")

verloop.plot(x="jaar", y=["nieuwe_leden", "vertrokken_leden"], marker="o", figsize=(10, 4),
             title="Nieuwe en vertrokken commissieleden per jaar")
plt.show()

In [ ]:
zittingsduur = query_df("""
  SELECT
    c.naam_nl AS naam,
    COUNT(*) AS aantal_zittingen,
    ROUND(AVG(date_diff('day', CAST(zp.van AS DATE),
                        COALESCE(CAST(zp.tot_en_met AS DATE), current_date))) / 365.25, 1) AS gem_jaren
  FROM silver.commissie_zetel_vast_persoon zp
  JOIN silver.commissie_zetel z ON z.id = zp.commissie_zetel_id
  JOIN silver.commissie c ON c.id = z.commissie_id
  WHERE zp.van IS NOT NULL
  GROUP BY c.naam_nl
  HAVING COUNT(*) >= 10
  ORDER BY gem_jaren
  LIMIT 20
""")

zittingsduur.plot.barh(x="naam", y="gem_jaren", figsize=(10, 7), legend=False,
                       title="20 commissies met de kortste gemiddelde zittingsduur (jaren, min. 10 zittingen)")
plt.show()

## 5. Geslachtsverhouding binnen de commissies
We onderzoeken hoe de verhouding tussen mannen en vrouwen is binnen de huidige commissies. We gebruiken alleen huidige leden van actieve commissies en koppelen deze aan persoon om het geslacht te bepalen. De resultaten worden alleen getoond voor commissies met minimaal 5 leden, zodat kleine commissies minder zwaar meetellen.

In [ ]:
query_df("""
  SELECT geslacht, COUNT(*) AS aantal
  FROM silver.persoon
  GROUP BY geslacht
  ORDER BY aantal DESC
""")

In [ ]:
geslacht = query_df("""
  SELECT
    c.naam_nl AS naam,
    COUNT(DISTINCT p.id) FILTER (WHERE p.geslacht ILIKE 'vrouw%') AS vrouwen,
    COUNT(DISTINCT p.id) FILTER (WHERE p.geslacht ILIKE 'man%') AS mannen,
    COUNT(DISTINCT p.id) AS totaal
  FROM silver.commissie_zetel_vast_persoon zp
  JOIN silver.commissie_zetel z ON z.id = zp.commissie_zetel_id
  JOIN silver.commissie c ON c.id = z.commissie_id
  JOIN silver.persoon p ON p.id = zp.persoon_id
  WHERE zp.tot_en_met IS NULL AND c.datum_inactief IS NULL
  GROUP BY c.naam_nl
  HAVING COUNT(DISTINCT p.id) >= 5
""")

geslacht["pct_vrouw"] = (100 * geslacht["vrouwen"] / geslacht["totaal"]).round(1)
geslacht = geslacht.sort_values("pct_vrouw")

geslacht.plot.barh(x="naam", y="pct_vrouw", figsize=(10, 10), legend=False,
                   title="Percentage vrouwen per commissie (huidige leden)")
plt.axvline(50, color="red", linestyle="--")
plt.show()
geslacht.describe()

### Wie zijn de voorzitters?

**Vraag:** zijn voorzitters vaker man dan vrouw, vergeleken met gewone leden?

In [ ]:
functies = query_df("""
  SELECT
    zp.functie,
    COUNT(*) FILTER (WHERE p.geslacht ILIKE 'vrouw%') AS vrouwen,
    COUNT(*) FILTER (WHERE p.geslacht ILIKE 'man%') AS mannen,
    COUNT(*) AS totaal
  FROM silver.commissie_zetel_vast_persoon zp
  JOIN silver.persoon p ON p.id = zp.persoon_id
  WHERE zp.tot_en_met IS NULL
  GROUP BY zp.functie
  ORDER BY totaal DESC
""")

functies.plot.bar(x="functie", y=["vrouwen", "mannen"], stacked=True, figsize=(10, 4),
                  title="Huidige commissieleden per functie en geslacht")
plt.show()
functies

## 6. Leeftijdsverdeling binnen de commissies
We onderzoeken of de verdeling tussen mannen en vrouwen bij voorzitters verschilt van die bij gewone commissieleden. Hiervoor vergelijken we de beschikbare gegevens over voorzitters met de geslachtsverdeling binnen de commissies.

In [ ]:
leeftijden = query_df("""
  SELECT DISTINCT
    c.naam_nl AS naam,
    p.id AS persoon_id,
    date_sub('year', CAST(p.geboortedatum AS DATE), current_date) AS leeftijd
  FROM silver.commissie_zetel_vast_persoon zp
  JOIN silver.commissie_zetel z ON z.id = zp.commissie_zetel_id
  JOIN silver.commissie c ON c.id = z.commissie_id
  JOIN silver.persoon p ON p.id = zp.persoon_id
  WHERE zp.tot_en_met IS NULL
    AND c.datum_inactief IS NULL
    AND p.geboortedatum IS NOT NULL
""")

leeftijden["leeftijd"].plot.hist(bins=20, figsize=(8, 4), title="Leeftijd van huidige commissieleden")
plt.xlabel("leeftijd")
plt.show()

In [ ]:
top15 = leeftijden["naam"].value_counts().head(15).index
leeftijden[leeftijden["naam"].isin(top15)].boxplot(column="leeftijd", by="naam", vert=False, figsize=(10, 8))
plt.suptitle("")
plt.title("Leeftijdsverdeling in de 15 grootste commissies")
plt.show()

(leeftijden.groupby("naam")["leeftijd"]
    .agg(leden="count", gemiddeld="mean", mediaan="median", jongste="min", oudste="max")
    .round(1)
    .sort_values("gemiddeld"))

## 7. Welke commissies zijn het meest actief?
We onderzoeken welke commissies het vaakst als voortouwcommissie bij activiteiten optreden. Hiervoor gebruiken we de tabel activiteit en tellen we per commissie het aantal activiteiten. Zo kunnen we zien welke commissies het meest betrokken zijn bij activiteiten.

In [ ]:
query_df("""
  SELECT column_name
  FROM information_schema.columns
  WHERE table_schema = 'silver' AND table_name = 'activiteit'
    AND (column_name ILIKE '%voortouw%' OR column_name ILIKE '%commissie%')
""")

In [ ]:
VOORTOUW = "voortouwnaam"  # pas aan als de kolom in de uitvoer hierboven anders heet

voortouw = query_df(f"""
  SELECT
    {VOORTOUW} AS voortouwcommissie,
    COUNT(*) AS aantal_activiteiten
  FROM silver.activiteit
  WHERE {VOORTOUW} IS NOT NULL
  GROUP BY 1
  ORDER BY aantal_activiteiten DESC
  LIMIT 20
""")

voortouw.sort_values("aantal_activiteiten").plot.barh(x="voortouwcommissie", y="aantal_activiteiten",
                                                      figsize=(10, 7), legend=False,
                                                      title="Top 20 voortouwcommissies naar aantal activiteiten")
plt.show()

 ### 7b. Meest actieve vaste en tijdelijke commissies

We vergelijken de activiteit van vaste en tijdelijke commissies afzonderlijk. Daarbij koppelen we via voortouwcommissie_id aan commissie, zodat TK niet als commissie wordt meegerekend. Zo ontstaat een eerlijker beeld van welke vaste en tijdelijke commissies het meest actief zijn.

In [ ]:
activiteiten = query_df(f"""
  SELECT
    c.naam_nl AS naam,
    {COMMISSIE_TYPE} AS commissietype,
    COUNT(*) AS aantal_activiteiten
  FROM silver.activiteit a
  JOIN silver.commissie c ON c.id = a.voortouwcommissie_id
  GROUP BY ALL
  ORDER BY aantal_activiteiten DESC
""")

for soort in ["vast", "tijdelijk"]:
    deel = activiteiten[activiteiten["commissietype"] == soort].head(10)
    if deel.empty:
        print(f"Geen commissies van type '{soort}' met activiteiten.")
        continue
    deel.sort_values("aantal_activiteiten").plot.barh(
        x="naam", y="aantal_activiteiten", figsize=(10, 5), legend=False,
        title=f"Top 10 commissies van type '{soort}' naar aantal activiteiten")
    plt.show()

## 7C
We onderzoeken ook hoeveel zaken aan commissies gekoppeld zijn. Hiervoor volgen we de keten commissie → activiteit → agendapunt → besluit → besluit_zaak. Alleen zaken waarover een besluit is genomen tellen mee. Hierbij zijn de exacte kolomnamen van enkele tabellen een aandachtspunt, omdat deze niet vooraf zijn gecontroleerd.

In [ ]:
zaken = query_df(f"""
  SELECT
    c.naam_nl AS naam,
    {COMMISSIE_TYPE} AS commissietype,
    COUNT(DISTINCT bz.zaak_id) AS aantal_zaken
  FROM silver.activiteit a
  JOIN silver.commissie c ON c.id = a.voortouwcommissie_id
  JOIN silver.agendapunt ap ON ap.activiteit_id = a.id
  JOIN silver.besluit b ON b.agendapunt_id = ap.id
  JOIN silver.besluit_zaak bz ON bz.besluit_id = b.id
  GROUP BY ALL
  ORDER BY aantal_zaken DESC
""")

zaken.head(20).sort_values("aantal_zaken").plot.barh(
    x="naam", y="aantal_zaken", figsize=(10, 8), legend=False,
    title="Top 20 commissies naar aantal verschillende zaken met een besluit")
plt.show()

## 8. Welke commissies zijn het meest effectief?
We onderzoeken welke commissies het meest effectief lijken op basis van de beschikbare gegevens. Omdat effectiviteit niet direct meetbaar is, kijken we naar het aantal activiteiten en naar het aandeel activiteiten dat is uitgevoerd tegenover geannuleerde of vervallen activiteiten. Dit geeft een indicatie van effectiviteit, maar is geen direct bewijs dat een commissie beter werkt dan een andere.

In [ ]:
query_df("""
  SELECT status, COUNT(*) AS aantal
  FROM silver.activiteit
  GROUP BY status
  ORDER BY aantal DESC
""")

In [ ]:
effectief = query_df(f"""
  SELECT
    {VOORTOUW} AS voortouwcommissie,
    COUNT(*) AS totaal,
    COUNT(*) FILTER (WHERE status ILIKE 'uitgevoerd') AS uitgevoerd,
    ROUND(100.0 * COUNT(*) FILTER (WHERE status ILIKE 'uitgevoerd') / COUNT(*), 1) AS pct_uitgevoerd
  FROM silver.activiteit
  WHERE {VOORTOUW} IS NOT NULL
  GROUP BY 1
  HAVING COUNT(*) >= 100
  ORDER BY pct_uitgevoerd DESC
""")

effectief.sort_values("pct_uitgevoerd").plot.barh(x="voortouwcommissie", y="pct_uitgevoerd",
                                                  figsize=(10, 8), legend=False,
                                                  title="Percentage uitgevoerde activiteiten per voortouwcommissie (min. 100)")
plt.show()
effectief

## 9. Datakwaliteit
We controleren de data op mogelijke onvolkomenheden en opvallende waarden. Zo kunnen we bepalen waar ontbrekende of afwijkende gegevens voorkomen en of deze mogelijk door de brondata of door het ETL-proces zijn ontstaan. Uit de controle blijkt dat er geen ontbrekende gegevens zijn gevonden.

In [ ]:
kwaliteit = query_df("""
  SELECT
    (SELECT COUNT(*) - COUNT(DISTINCT id) FROM silver.commissie_zetel_vast_persoon) AS dubbele_id_zitting,
    (SELECT COUNT(*) FROM silver.commissie_zetel_vast_persoon WHERE tot_en_met < van) AS zitting_einde_voor_start,
    (SELECT COUNT(*) FROM silver.commissie_zetel_vast_persoon WHERE persoon_id IS NULL) AS zitting_zonder_persoon,
    (SELECT COUNT(*) FROM silver.commissie_zetel_vast_vacature WHERE tot_en_met < van) AS vacature_einde_voor_start,
    (SELECT COUNT(*) FROM silver.persoon WHERE geboortedatum IS NULL) AS persoon_zonder_geboortedatum,
    (SELECT COUNT(*) FROM silver.persoon WHERE geslacht IS NULL) AS persoon_zonder_geslacht,
    (SELECT COUNT(*) FROM silver.commissie_zetel z
       LEFT JOIN silver.commissie c ON c.id = z.commissie_id WHERE c.id IS NULL) AS zetel_zonder_commissie
""")
kwaliteit.T.rename(columns={0: "aantal"})

In [ ]:
# Onwaarschijnlijke leeftijden onder de huidige commissieleden
leeftijden[(leeftijden["leeftijd"] < 18) | (leeftijden["leeftijd"] > 90)]

## Conclusies